In [1]:
from sklearn.model_selection import LeaveOneGroupOut, StratifiedKFold, train_test_split
import sys
import os

sys.path.append(os.path.abspath(os.path.join('..')))

from src.load_data_BCICIV import load_all_subjects
from src.preprocess import laplacian_filter
from src.preprocess import normalize_trial
from models.EEGNet import EEGNet
from torch.utils.data import DataLoader, Dataset, Subset
import torch
import torch.nn as nn
import torch.optim as optim
from torch.autograd import Variable
import torch.nn.functional as F
import numpy as np


In [2]:
class EEGDataset(Dataset):
    def __init__(self, X, y, transform=None):
        self.X = X
        self.y = y
        self.transform = transform

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = normalize_trial(self.X[idx])
        x = torch.tensor(x, dtype=torch.float32)
        
        y = torch.tensor(self.y[idx], dtype=torch.long)
        if self.transform:
            x = self.transform(x)
        
        return x, y


In [3]:
data_path = '../datasets/BCICIV_2a_gdf'

data = load_all_subjects(data_path)

/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Loaded file: ../datasets/BCICIV_2a_gdf/A01T.gdf


/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Loaded file: ../datasets/BCICIV_2a_gdf/A02T.gdf


/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


...

Loaded file: ../datasets/BCICIV_2a_gdf/A07T.gdf


/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Loaded file: ../datasets/BCICIV_2a_gdf/A08T.gdf


/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Loaded file: ../datasets/BCICIV_2a_gdf/A09T.gdf


In [4]:
split_index_test = 1150
split_index_val = 862 

dataset = EEGDataset(data['X'], data['y'])
train_EEG = Subset(dataset, indices=range(0, split_index_val))
val_EEG = Subset(dataset, indices=range(split_index_val, split_index_test))
test_EEG = Subset(dataset, indices=range(split_index_test, len(dataset)))

train_dl = DataLoader(train_EEG, batch_size=32, shuffle=True)
val_dl = DataLoader(val_EEG, batch_size=32, shuffle=False)
test_dl = DataLoader(test_EEG, batch_size=32, shuffle=False)

In [5]:
def train_model(model, train_dl, val_dl, epochs=100, lr=0.0005, patience=20):

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    best_val_acc = 0
    best_model_state = None
    patience_counter = 0

    for epoch in range(epochs):

        model.train()
        train_loss = 0.0
        train_correct = 0

        for batch_x, batch_y in train_dl:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)

            optimizer.zero_grad()
            output = model(batch_x)
            loss = criterion(output, batch_y)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * batch_x.size(0)
            _, predicted = torch.max(output, 1)
            train_correct += (predicted == batch_y).sum().item()

        train_loss /= len(train_dl.dataset)
        train_acc = train_correct / len(train_dl.dataset)

        model.eval()
        val_loss = 0.0
        val_correct = 0

        with torch.no_grad():
            for batch_x, batch_y in val_dl:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)

                output = model(batch_x)
                loss = criterion(output, batch_y)

                val_loss += loss.item() * batch_x.size(0)
                _, predicted = torch.max(output, 1)
                val_correct += (predicted == batch_y).sum().item()

        val_loss /= len(val_dl.dataset)
        val_acc = val_correct / len(val_dl.dataset)

        print(f"Epoch {epoch+1}: "
              f"Train Loss {train_loss:.4f} | Train Acc {train_acc:.4f} | "
              f"Val Loss {val_loss:.4f} | Val Acc {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print("Early stopping.")
            break

    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    return model

In [6]:
def init_weights_he(m):
    if isinstance(m, nn.Conv2d):
        nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
        if m.bias is not None:
            nn.init.zeros_(m.bias)

In [7]:
model = EEGNet(11, 1000, 2, f1=8, D=2, dropout_rate=0.5)
model.apply(init_weights_he)

trained_model = train_model(model, train_dl, val_dl, epochs=100, lr=0.0005, patience=20)

Epoch 1: Train Loss 0.6927 | Train Acc 0.5371 | Val Loss 0.6890 | Val Acc 0.5347
Epoch 2: Train Loss 0.6989 | Train Acc 0.5058 | Val Loss 0.6866 | Val Acc 0.5278
Epoch 3: Train Loss 0.6961 | Train Acc 0.5116 | Val Loss 0.6861 | Val Acc 0.5208
Epoch 4: Train Loss 0.6926 | Train Acc 0.5278 | Val Loss 0.6843 | Val Acc 0.5521
Epoch 5: Train Loss 0.6837 | Train Acc 0.5406 | Val Loss 0.6828 | Val Acc 0.5764
Epoch 6: Train Loss 0.6909 | Train Acc 0.5244 | Val Loss 0.6806 | Val Acc 0.5729
Epoch 7: Train Loss 0.6923 | Train Acc 0.5093 | Val Loss 0.6797 | Val Acc 0.5868
Epoch 8: Train Loss 0.6831 | Train Acc 0.5545 | Val Loss 0.6785 | Val Acc 0.5903
Epoch 9: Train Loss 0.6877 | Train Acc 0.5452 | Val Loss 0.6786 | Val Acc 0.5625
Epoch 10: Train Loss 0.6855 | Train Acc 0.5441 | Val Loss 0.6782 | Val Acc 0.5972
Epoch 11: Train Loss 0.6767 | Train Acc 0.5754 | Val Loss 0.6784 | Val Acc 0.5833
Epoch 12: Train Loss 0.6704 | Train Acc 0.5974 | Val Loss 0.6773 | Val Acc 0.6042
Epoch 13: Train Loss 0.67